# 03a - Model training: 8 architectures, 4 encoder sources
Trains and compares 8 segmentation architectures on the Sentinel-1 SAR dataset from `02_preprocessing_32cities_update.ipynb`.

**Cell 1:** Install libraries, mount Drive, copy patches to local disk for fast reads.

In [5]:
!pip install -q segmentation-models-pytorch torchgeo rasterio

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import time

DRIVE_FOLDER = '/content/drive/MyDrive/lulc_dataset'
DRIVE_PATCH_FOLDER = f'{DRIVE_FOLDER}/patches'
PATCH_FOLDER = '/content/local_patches'
CKPT_DIR = f'{DRIVE_FOLDER}/All Model Weights'
os.makedirs(CKPT_DIR, exist_ok=True)

# Copy patches to local disk once. Reading from Drive file-by-file later on
# is much slower than reading from local disk. Copies one file at a time in
# Python (not a shell `cp`) so progress prints live instead of going silent
# until the whole copy finishes.
os.makedirs(PATCH_FOLDER, exist_ok=True)
drive_files = os.listdir(DRIVE_PATCH_FOLDER)
already_copied = set(os.listdir(PATCH_FOLDER))
to_copy = [f for f in drive_files if f not in already_copied]

if to_copy:
    print(f'Copying {len(to_copy)} of {len(drive_files)} patch files to local disk...')
    t0 = time.time()
    for n, fname in enumerate(to_copy):
        shutil.copy2(f'{DRIVE_PATCH_FOLDER}/{fname}', f'{PATCH_FOLDER}/{fname}')
        if (n + 1) % 500 == 0 or (n + 1) == len(to_copy):
            elapsed = time.time() - t0
            rate = (n + 1) / elapsed if elapsed > 0 else 0
            eta = (len(to_copy) - (n + 1)) / rate if rate > 0 else 0
            print(f'  {n+1}/{len(to_copy)} files copied ({elapsed:.0f}s elapsed, ~{eta:.0f}s left)')
    print('Copy done.')
else:
    print(f'All {len(drive_files)} patch files already on local disk, skipping copy.')

print('Drive patch folder:', DRIVE_PATCH_FOLDER)
print('Local patch folder (used for training):', PATCH_FOLDER)
print('Checkpoint folder:', CKPT_DIR)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
Copying 25062 of 25062 patch files to local disk...
  500/25062 files copied (410s elapsed, ~20140s left)
  1000/25062 files copied (788s elapsed, ~18954s left)
  1500/25062 files copied (1179s elapsed, ~18516s left)
  2000/25062 files copied (1553s elapsed, ~17904s left)
  2500/25062 files copied (1945s elapsed, ~17554s left)
  3000/25062 files copied (2324s elapsed, ~17092s left)
  3500/25062 files copied (2724s elapsed, ~16781s left)
  4000/25062 files copied (3111s elapsed, ~16381s left)
  4500/25062 files copied (3501s elapsed, ~15997s left)
  5000/25062 files copied (3873s elapsed, ~15541s left)
  5500/25062 files copied (4254s elapsed, ~15130s left)
  6000/25062 files copied (4614s elapsed, ~14658s left)
  6500/25062 files copied (4996s elapsed, ~14266s left)
  7000/25062 files copied (5351s elapsed, ~13808s left)
  7500

In [6]:
import shutil
import subprocess
import time

t0 = time.time()
subprocess.run(['zip', '-r', '-q', '/content/patches_backup.zip', '.'], cwd='/content/local_patches')
print(f'Zipped locally in {time.time()-t0:.0f}s')

t0 = time.time()
shutil.copy('/content/patches_backup.zip', '/content/drive/MyDrive/lulc_dataset/patches_backup.zip')
print(f'Uploaded zip to Drive in {time.time()-t0:.0f}s')

Zipped locally in 516s
Uploaded zip to Drive in 24s


**Cell 2:** Load patch metadata, split into train/val/3 test sets.

In [7]:
import pandas as pd

patch_df = pd.read_csv(f'{DRIVE_FOLDER}/patch_metadata.csv')
print(f'Loaded {len(patch_df)} total patch records')
print(patch_df['split'].value_counts())

train_df = patch_df[patch_df['split'] == 'train'].reset_index(drop=True)
val_df = patch_df[patch_df['split'] == 'val'].reset_index(drop=True)
test_same_region_df = patch_df[patch_df['split'] == 'test_same_region'].reset_index(drop=True)
test_similar_unseen_df = patch_df[patch_df['split'] == 'test_similar_unseen'].reset_index(drop=True)
test_dissimilar_df = patch_df[patch_df['split'] == 'test_dissimilar'].reset_index(drop=True)

print(f'\ntrain={len(train_df)} val={len(val_df)} test_same_region={len(test_same_region_df)} '
      f'test_similar_unseen={len(test_similar_unseen_df)} test_dissimilar={len(test_dissimilar_df)}')


Loaded 12288 total patch records
split
train                  7680
test_dissimilar        1536
test_similar_unseen    1152
unassigned              768
test_same_region        768
val                     384
Name: count, dtype: int64

train=7680 val=384 test_same_region=768 test_similar_unseen=1152 test_dissimilar=1536


**Cell 3:** Define the 9 real classes and remap raw pixel codes to 0-8.

In [8]:
import numpy as np

RAW_CODES = [10, 20, 30, 40, 50, 60, 80, 90, 95]
CLASS_NAMES = [
    "Tree cover", "Shrubland", "Grassland", "Cropland", "Built-up",
    "Bare/sparse vegetation", "Permanent water", "Herbaceous wetland", "Mangroves",
]
NUM_CLASSES = len(RAW_CODES)
IGNORE_INDEX = 255
RARE_CLASSES = ["Shrubland", "Bare/sparse vegetation", "Herbaceous wetland", "Mangroves"]

_LUT = np.full(256, IGNORE_INDEX, dtype=np.uint8)
for i, code_val in enumerate(RAW_CODES):
    _LUT[code_val] = i

def remap_label(label_array):
    label_array = np.clip(label_array.astype(np.int64), 0, 255)
    return _LUT[label_array]

print('Classes used for training:', CLASS_NAMES)


Classes used for training: ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Bare/sparse vegetation', 'Permanent water', 'Herbaceous wetland', 'Mangroves']


**Cell 4:** Flag rare-class patches, compute class weights, define the dataset class. Both scanning steps print progress every 200 patches with an ETA.

In [9]:
import time
import torch
from torch.utils.data import Dataset

def _progress(n, total, t0, every=200):
    if (n + 1) % every == 0 or (n + 1) == total:
        elapsed = time.time() - t0
        rate = (n + 1) / elapsed if elapsed > 0 else 0
        eta = (total - (n + 1)) / rate if rate > 0 else 0
        print(f'  {n+1}/{total} patches scanned ({elapsed:.0f}s elapsed, ~{eta:.0f}s left)')

def tag_rare_patches(df, patch_folder, label_col='label_patch'):
    rare_ids = {i for i, name in enumerate(CLASS_NAMES) if name in RARE_CLASSES}
    flags = []
    t0 = time.time()
    for n, (_, row) in enumerate(df.iterrows()):
        lbl = np.load(f'{patch_folder}/{row[label_col]}')
        remapped = remap_label(lbl)
        present = set(np.unique(remapped).tolist())
        flags.append(bool(present & rare_ids))
        _progress(n, len(df), t0)
    out = df.copy()
    out['is_rare'] = flags
    return out

def compute_class_weights(df, patch_folder, label_col='label_patch'):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    t0 = time.time()
    for n, (_, row) in enumerate(df.iterrows()):
        lbl = np.load(f'{patch_folder}/{row[label_col]}')
        remapped = remap_label(lbl)
        values, c = np.unique(remapped, return_counts=True)
        for v, cnt in zip(values, c):
            if v != IGNORE_INDEX:
                counts[v] += cnt
        _progress(n, len(df), t0)
    counts = np.maximum(counts, 1)
    weights = 1.0 / np.sqrt(counts)
    weights = weights / weights.sum() * NUM_CLASSES
    weights = np.clip(weights, None, 10.0)
    return torch.tensor(weights, dtype=torch.float32)


class PatchDataset(Dataset):
    """add_vv_vh_diff=True appends a 3rd channel (VV-VH) for SegNet/FCDenseNet,
    which expect 3-channel ImageNet-style input. The 6 ResNet-50 models stay
    2-channel to match their pretrained first convolution."""

    def __init__(self, df, patch_folder, augment=False, oversample_rare=False,
                 image_col='image_patch', label_col='label_patch', add_vv_vh_diff=False):
        self.patch_folder = patch_folder
        self.augment = augment
        self.image_col = image_col
        self.label_col = label_col
        self.add_vv_vh_diff = add_vv_vh_diff

        if oversample_rare:
            if 'is_rare' not in df.columns:
                raise ValueError("oversample_rare=True needs tag_rare_patches() run on this df first.")
            rare_rows = df[df['is_rare']]
            if len(rare_rows) > 0:
                df = pd.concat([df, rare_rows, rare_rows], ignore_index=True)
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        img = np.load(f'{self.patch_folder}/{row[self.image_col]}').astype(np.float32)
        lbl_raw = np.load(f'{self.patch_folder}/{row[self.label_col]}')
        lbl = remap_label(lbl_raw)

        if self.add_vv_vh_diff:
            if img.shape[0] != 2:
                raise ValueError(f'add_vv_vh_diff expects a 2-channel patch, got shape {img.shape}')
            vv_vh_diff = (img[0] - img[1])[np.newaxis, :, :]
            img = np.concatenate([img, vv_vh_diff], axis=0)

        if self.augment:
            if np.random.rand() > 0.5:
                img = np.flip(img, axis=2).copy()
                lbl = np.flip(lbl, axis=1).copy()
            if np.random.rand() > 0.5:
                img = np.flip(img, axis=1).copy()
                lbl = np.flip(lbl, axis=0).copy()
            k = np.random.choice([0, 1, 2, 3])
            img = np.rot90(img, k, axes=(1, 2)).copy()
            lbl = np.rot90(lbl, k, axes=(0, 1)).copy()

        return torch.from_numpy(img).float(), torch.from_numpy(lbl).long()


print('Tagging rare-class patches in the training split...')
train_df = tag_rare_patches(train_df, PATCH_FOLDER)
print(f'{train_df["is_rare"].sum()} of {len(train_df)} training patches contain a rare class.')

print('Computing class weights from the training split...')
class_weights = compute_class_weights(train_df, PATCH_FOLDER)
for name, w in zip(CLASS_NAMES, class_weights.tolist()):
    print(f'  {name}: {w:.3f}')


Tagging rare-class patches in the training split...
  200/7680 patches scanned (0s elapsed, ~15s left)
  400/7680 patches scanned (1s elapsed, ~13s left)
  600/7680 patches scanned (1s elapsed, ~12s left)
  800/7680 patches scanned (1s elapsed, ~11s left)
  1000/7680 patches scanned (2s elapsed, ~11s left)
  1200/7680 patches scanned (2s elapsed, ~10s left)
  1400/7680 patches scanned (2s elapsed, ~10s left)
  1600/7680 patches scanned (3s elapsed, ~10s left)
  1800/7680 patches scanned (3s elapsed, ~9s left)
  2000/7680 patches scanned (3s elapsed, ~9s left)
  2200/7680 patches scanned (3s elapsed, ~9s left)
  2400/7680 patches scanned (4s elapsed, ~9s left)
  2600/7680 patches scanned (4s elapsed, ~8s left)
  2800/7680 patches scanned (4s elapsed, ~8s left)
  3000/7680 patches scanned (5s elapsed, ~7s left)
  3200/7680 patches scanned (5s elapsed, ~7s left)
  3400/7680 patches scanned (5s elapsed, ~7s left)
  3600/7680 patches scanned (6s elapsed, ~6s left)
  3800/7680 patches scanne

**Cell 5:** Loss function: class-weighted Cross-Entropy + Dice.

In [10]:
class CombinedLoss(torch.nn.Module):
    def __init__(self, class_weights=None, ignore_index=IGNORE_INDEX, dice_weight=0.5):
        super().__init__()
        self.ce = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        self.ignore_index = ignore_index
        self.dice_weight = dice_weight
        self.num_classes = NUM_CLASSES

    def dice_loss(self, logits, target):
        valid = (target != self.ignore_index)
        target_clamped = target.clone()
        target_clamped[~valid] = 0

        probs = torch.softmax(logits, dim=1)
        target_onehot = torch.nn.functional.one_hot(target_clamped, self.num_classes).permute(0, 3, 1, 2).float()

        valid_f = valid.unsqueeze(1).float()
        probs = probs * valid_f
        target_onehot = target_onehot * valid_f

        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_onehot, dims)
        cardinality = torch.sum(probs + target_onehot, dims)
        dice_per_class = (2.0 * intersection + 1e-6) / (cardinality + 1e-6)
        return 1.0 - dice_per_class.mean()

    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)
        d_loss = self.dice_loss(logits, target)
        return (1 - self.dice_weight) * ce_loss + self.dice_weight * d_loss


loss_fn = CombinedLoss(class_weights=class_weights)
print('Loss function ready: class-weighted Cross-Entropy + Dice, ignore_index =', IGNORE_INDEX)


Loss function ready: class-weighted Cross-Entropy + Dice, ignore_index = 255


**Cell 6:** Metrics: mIoU, per-class IoU/F1, macro-F1, kappa (ignore index excluded).

In [11]:
def compute_confusion_matrix(preds, targets, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    if torch.is_tensor(preds):
        preds = preds.cpu().numpy()
    if torch.is_tensor(targets):
        targets = targets.cpu().numpy()
    valid = targets != ignore_index
    preds = preds[valid]
    targets = targets[valid]
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    idx = targets * num_classes + preds
    binc = np.bincount(idx, minlength=num_classes * num_classes)
    cm += binc[: num_classes * num_classes].reshape(num_classes, num_classes)
    return cm


def metrics_from_confusion_matrix(cm):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    total = cm.sum()

    oa = tp.sum() / total if total > 0 else 0.0

    union = tp + fp + fn
    iou = np.divide(tp, union, out=np.zeros_like(tp), where=union > 0)
    present = union > 0
    miou = iou[present].mean() if present.any() else 0.0

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                    out=np.zeros_like(tp), where=(precision + recall) > 0)
    macro_f1 = f1[present].mean() if present.any() else 0.0

    row_sum = cm.sum(axis=1)
    col_sum = cm.sum(axis=0)
    expected = (row_sum * col_sum).sum() / total if total > 0 else 0.0
    kappa = (tp.sum() - expected) / (total - expected) if (total - expected) != 0 else 0.0

    return {
        'OA': float(oa), 'mIoU': float(miou), 'macro_F1': float(macro_f1), 'kappa': float(kappa),
        'per_class_IoU': iou.tolist(), 'per_class_F1': f1.tolist(),
    }

print('Metrics functions ready.')


Metrics functions ready.


**Cell 7:** SegNet architecture (VGG-16 encoder, max-unpooling decoder, 3-channel input).

In [12]:
import torchvision

class SegNet(torch.nn.Module):
    def __init__(self, in_channels=3, num_classes=9, pretrained=True):
        super().__init__()
        weights = torchvision.models.VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None
        vgg16 = torchvision.models.vgg16_bn(weights=weights)
        features = list(vgg16.features.children())

        if in_channels != 3:
            old_conv = features[0]
            new_conv = torch.nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size, stride=old_conv.stride, padding=old_conv.padding,
            )
            if pretrained:
                with torch.no_grad():
                    if in_channels <= 3:
                        new_conv.weight[:] = old_conv.weight[:, :in_channels, :, :].clone()
                    else:
                        reps = (in_channels + 2) // 3
                        tiled = old_conv.weight.repeat(1, reps, 1, 1)[:, :in_channels, :, :]
                        new_conv.weight[:] = tiled.clone()
            features[0] = new_conv

        self.encoder_stages = torch.nn.ModuleList()
        stage_layers = []
        for layer in features:
            if isinstance(layer, torch.nn.MaxPool2d):
                self.encoder_stages.append(torch.nn.Sequential(*stage_layers))
                stage_layers = []
            else:
                stage_layers.append(layer)
        assert len(self.encoder_stages) == 5
        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)

        decoder_configs = [(512, 512, 3), (512, 256, 3), (256, 128, 3), (128, 64, 2), (64, 64, 2)]
        self.decoder_stages = torch.nn.ModuleList()
        for in_c, out_c, n_convs in decoder_configs:
            layers = []
            ch = in_c
            for i in range(n_convs):
                out = out_c if i == n_convs - 1 else ch
                layers += [torch.nn.Conv2d(ch, out, 3, padding=1), torch.nn.BatchNorm2d(out), torch.nn.ReLU(inplace=True)]
                ch = out
            self.decoder_stages.append(torch.nn.Sequential(*layers))

        self.final_conv = torch.nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        pool_indices = []
        pre_pool_sizes = []
        for stage in self.encoder_stages:
            x = stage(x)
            pre_pool_sizes.append(x.size())
            x, idx = self.pool(x)
            pool_indices.append(idx)

        for i, stage in enumerate(self.decoder_stages):
            idx = pool_indices[-(i + 1)]
            size = pre_pool_sizes[-(i + 1)]
            x = torch.nn.functional.max_unpool2d(x, idx, kernel_size=2, stride=2, output_size=size)
            x = stage(x)

        return self.final_conv(x)

print('SegNet class ready.')


SegNet class ready.


**Cell 8:** FC-DenseNet architecture (dense blocks, trained from scratch, 3-channel input).

In [13]:
class DenseLayer(torch.nn.Module):
    def __init__(self, in_channels, growth_rate, dropout=0.2):
        super().__init__()
        self.bn = torch.nn.BatchNorm2d(in_channels)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv = torch.nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        self.drop = torch.nn.Dropout2d(dropout)

    def forward(self, x):
        return self.drop(self.conv(self.relu(self.bn(x))))


class DenseBlock(torch.nn.Module):
    def __init__(self, in_channels, growth_rate, n_layers, dropout=0.2):
        super().__init__()
        self.layers = torch.nn.ModuleList([
            DenseLayer(in_channels + i * growth_rate, growth_rate, dropout)
            for i in range(n_layers)
        ])

    def forward(self, x):
        new_features = []
        for layer in self.layers:
            out = layer(x)
            new_features.append(out)
            x = torch.cat([x, out], dim=1)
        new_only = torch.cat(new_features, dim=1)
        return x, new_only


class TransitionDown(torch.nn.Module):
    def __init__(self, channels, dropout=0.2):
        super().__init__()
        self.bn = torch.nn.BatchNorm2d(channels)
        self.relu = torch.nn.ReLU(inplace=True)
        self.conv = torch.nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.drop = torch.nn.Dropout2d(dropout)
        self.pool = torch.nn.MaxPool2d(2)

    def forward(self, x):
        x = self.drop(self.conv(self.relu(self.bn(x))))
        return self.pool(x)


class TransitionUp(torch.nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.convT = torch.nn.ConvTranspose2d(channels, channels, kernel_size=3, stride=2, padding=1, output_padding=1)

    def forward(self, x, target_size):
        x = self.convT(x)
        if x.shape[-2:] != target_size:
            x = torch.nn.functional.interpolate(x, size=target_size, mode='nearest')
        return x


class FCDenseNet56(torch.nn.Module):
    def __init__(self, in_channels=3, num_classes=9, growth_rate=12, n_layers_per_block=4,
                 n_blocks=5, n_init_features=48, dropout=0.2):
        super().__init__()
        self.init_conv = torch.nn.Conv2d(in_channels, n_init_features, kernel_size=3, padding=1, bias=False)

        cur_channels = n_init_features
        self.down_blocks = torch.nn.ModuleList()
        self.trans_down = torch.nn.ModuleList()
        skip_channels = []
        for _ in range(n_blocks):
            self.down_blocks.append(DenseBlock(cur_channels, growth_rate, n_layers_per_block, dropout))
            cur_channels = cur_channels + n_layers_per_block * growth_rate
            skip_channels.append(n_layers_per_block * growth_rate)
            self.trans_down.append(TransitionDown(cur_channels, dropout))

        self.bottleneck = DenseBlock(cur_channels, growth_rate, n_layers_per_block, dropout)
        up_channels = n_layers_per_block * growth_rate

        self.trans_up = torch.nn.ModuleList()
        self.up_blocks = torch.nn.ModuleList()
        for i in reversed(range(n_blocks)):
            self.trans_up.append(TransitionUp(up_channels))
            concat_channels = up_channels + skip_channels[i]
            self.up_blocks.append(DenseBlock(concat_channels, growth_rate, n_layers_per_block, dropout))
            up_channels = n_layers_per_block * growth_rate

        self.final_conv = torch.nn.Conv2d(up_channels, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.init_conv(x)

        skips = []
        skip_sizes = []
        for down_block, td in zip(self.down_blocks, self.trans_down):
            x, new_only = down_block(x)
            skips.append(new_only)
            skip_sizes.append(new_only.shape[-2:])
            x = td(x)

        _, x = self.bottleneck(x)

        for i, (tu, up_block) in enumerate(zip(self.trans_up, self.up_blocks)):
            skip = skips[-(i + 1)]
            size = skip_sizes[-(i + 1)]
            x = tu(x, size)
            x = torch.cat([x, skip], dim=1)
            _, x = up_block(x)

        return self.final_conv(x)

print('FCDenseNet56 class ready.')


FCDenseNet56 class ready.


**Cell 9:** `build_model()`: builds any of the 8 architectures, with 4 encoder options for the 6 ResNet-50-based ones.

In [14]:
import segmentation_models_pytorch as smp
import torchgeo.models as tgm

SMP_BUILDERS = {
    'Unet': smp.Unet, 'UnetPlusPlus': smp.UnetPlusPlus, 'MAnet': smp.MAnet,
    'DeepLabV3Plus': smp.DeepLabV3Plus, 'FPN': smp.FPN, 'PSPNet': smp.PSPNet,
}

TORCHGEO_SAR_WEIGHTS = {
    'ssl4eo_moco': 'SENTINEL1_ALL_MOCO',
    'softcon': 'SENTINEL1_GRD_SOFTCON',
}


def _load_resnet50_sar_encoder(smp_model, weights_name, freeze=False):
    """strict=True: nn.Module.load_state_dict always raises a RuntimeError on
    any real key mismatch, regardless of smp's own encoder override discarding
    the return value, so a genuine incompatibility still fails loudly."""
    weights_enum = getattr(tgm.ResNet50_Weights, weights_name)
    state_dict = weights_enum.get_state_dict(progress=True)
    smp_model.encoder.load_state_dict(state_dict, strict=True)
    if freeze:
        for p in smp_model.encoder.parameters():
            p.requires_grad = False
    return smp_model


def build_model(architecture, encoder_source, num_classes=NUM_CLASSES, in_channels=2, freeze_encoder=False):
    if architecture == 'SegNet':
        pretrained = encoder_source != 'random'
        return SegNet(in_channels=3, num_classes=num_classes, pretrained=pretrained)

    if architecture == 'FCDenseNet':
        return FCDenseNet56(in_channels=3, num_classes=num_classes)

    if architecture not in SMP_BUILDERS:
        raise ValueError(f'Unknown architecture: {architecture}')

    builder = SMP_BUILDERS[architecture]

    if encoder_source == 'imagenet':
        model = builder(encoder_name='resnet50', encoder_weights='imagenet',
                         in_channels=in_channels, classes=num_classes)
        if freeze_encoder:
            for p in model.encoder.parameters():
                p.requires_grad = False
        return model

    if encoder_source == 'random':
        return builder(encoder_name='resnet50', encoder_weights=None,
                        in_channels=in_channels, classes=num_classes)

    if encoder_source in TORCHGEO_SAR_WEIGHTS:
        model = builder(encoder_name='resnet50', encoder_weights=None,
                         in_channels=in_channels, classes=num_classes)
        return _load_resnet50_sar_encoder(model, TORCHGEO_SAR_WEIGHTS[encoder_source], freeze=freeze_encoder)

    raise ValueError(f'Unknown encoder_source: {encoder_source}')

print('build_model() ready. Architectures:', list(SMP_BUILDERS.keys()) + ['SegNet', 'FCDenseNet'])
print('Encoder sources for the 6 ResNet-50 models:', list(TORCHGEO_SAR_WEIGHTS.keys()) + ['imagenet', 'random'])


build_model() ready. Architectures: ['Unet', 'UnetPlusPlus', 'MAnet', 'DeepLabV3Plus', 'FPN', 'PSPNet', 'SegNet', 'FCDenseNet']
Encoder sources for the 6 ResNet-50 models: ['ssl4eo_moco', 'softcon', 'imagenet', 'random']


**Cell 10:** `train_one_run()`: trains, checkpoints the best epoch, early-stops, resumes if interrupted.

In [15]:
import gc
import time
import json as json_module
from torch.utils.data import DataLoader


def with_retries(func, *args, retries=3, delay=10, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except OSError as e:
            last_err = e
            print(f'    [retry {attempt}/{retries}] disk error: {e}')
            if attempt == retries:
                break
            time.sleep(delay)
    raise last_err


def make_optimizer_and_scheduler(model, encoder_lr, decoder_lr, total_steps, warmup_steps):
    encoder_params, other_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (encoder_params if name.startswith('encoder.') else other_params).append(p)

    param_groups = [{'params': other_params, 'lr': decoder_lr}]
    if encoder_params:
        param_groups.append({'params': encoder_params, 'lr': encoder_lr})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


@torch.no_grad()
def evaluate(model, loader, device, num_classes=NUM_CLASSES, label=None):
    model.eval()
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    n_batches = len(loader)
    t0 = time.time()
    for b, (imgs, lbls) in enumerate(loader):
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy().ravel()
        targets = lbls.numpy().ravel()
        cm += compute_confusion_matrix(preds, targets, num_classes=num_classes)
        # Progress every 20 batches, so a large test set never runs silently.
        if n_batches > 20 and ((b + 1) % 20 == 0 or (b + 1) == n_batches):
            tag = f'[{label}] ' if label else ''
            print(f'  {tag}eval batch {b+1}/{n_batches} ({time.time()-t0:.0f}s elapsed)')
    return metrics_from_confusion_matrix(cm), cm


def train_one_run(run_name, model, train_ds, val_ds, loss_fn_, device,
                   batch_size=8, max_epochs=30, patience=6,
                   encoder_lr=1e-5, decoder_lr=1e-4, warmup_epochs=2,
                   checkpoint_dir=CKPT_DIR, log_every=50):
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = f'{checkpoint_dir}/{run_name}_best.pt'
    log_path = f'{checkpoint_dir}/{run_name}_log.json'

    # num_workers=2: this specific runtime has 2 vCPUs (checked with os.cpu_count()), not 4.
    # Setting num_workers above the real core count doesn't help, the extra workers just
    # compete with the main process and each other for the same 2 cores, adding overhead
    # without any real parallelism gain. pin_memory=True speeds up the host-to-GPU transfer
    # for each batch once it's loaded.
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True,
                               num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False,
                             num_workers=2, pin_memory=True)

    steps_per_epoch = max(1, len(train_loader))
    total_steps = steps_per_epoch * max_epochs
    warmup_steps = steps_per_epoch * warmup_epochs

    # Move the model to the GPU BEFORE building the optimizer, not after. Building the optimizer
    # from model.parameters() first and only then calling .to(device) happens to still work under
    # PyTorch's current default settings (Parameter objects keep their identity across .to(), so the
    # optimizer's references stay valid), but that's a global default, not a guarantee. If it were ever
    # wrong, it wouldn't crash, it would just silently train stale CPU-side copies that the model no
    # longer uses, so loss would never actually move. Doing it in the documented-safe order instead.
    model = model.to(device)
    optimizer, scheduler = make_optimizer_and_scheduler(model, encoder_lr, decoder_lr, total_steps, warmup_steps)
    # CombinedLoss wraps nn.CrossEntropyLoss(weight=class_weights). class_weights is built as a
    # plain CPU tensor back in Cell 4, and nothing else in this pipeline ever moves the loss module
    # itself to the GPU, only the model. On a real CUDA run this crashes the moment the first batch
    # hits cross_entropy, since the logits are on cuda:0 but the loss's internal weight tensor is
    # still on cpu. Moving it here, once, covers every call site (Step A/B/C) without having to
    # remember to do it everywhere loss_fn is constructed.
    loss_fn_ = loss_fn_.to(device)
    best_miou = -1.0
    epochs_without_improvement = 0
    history = []
    start_epoch = 0

    if os.path.exists(log_path):
        with open(log_path) as f:
            history = json_module.load(f)
        if os.path.exists(ckpt_path) and history:
            print(f'[{run_name}] Resuming: {len(history)} epoch(s) already done.')
            model.load_state_dict(torch.load(ckpt_path, map_location=device))
            best_miou = max(h['val_mIoU'] for h in history)
            start_epoch = len(history)

    for epoch in range(start_epoch, max_epochs):
        model.train()
        running_loss, n_batches = 0.0, 0
        for step, (imgs, lbls) in enumerate(train_loader):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = loss_fn_(logits, lbls)
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()
            n_batches += 1
            if (step + 1) % log_every == 0:
                print(f'[{run_name}] epoch {epoch+1}/{max_epochs} step {step+1}/{steps_per_epoch} loss={running_loss/n_batches:.4f}')

        val_metrics, _ = evaluate(model, val_loader, device)
        avg_train_loss = running_loss / max(1, n_batches)
        print(f'[{run_name}] epoch {epoch+1}/{max_epochs} done. train_loss={avg_train_loss:.4f} '
              f'val_mIoU={val_metrics["mIoU"]:.4f} val_OA={val_metrics["OA"]:.4f}')

        history.append({
            'epoch': epoch + 1, 'train_loss': avg_train_loss,
            'val_OA': val_metrics['OA'], 'val_mIoU': val_metrics['mIoU'],
            'val_macro_F1': val_metrics['macro_F1'], 'val_kappa': val_metrics['kappa'],
            'val_per_class_IoU': val_metrics['per_class_IoU'], 'val_per_class_F1': val_metrics['per_class_F1'],
        })
        with_retries(lambda: json_module.dump(history, open(log_path, 'w'), indent=2))

        if val_metrics['mIoU'] > best_miou:
            best_miou = val_metrics['mIoU']
            epochs_without_improvement = 0
            with_retries(lambda: torch.save(model.state_dict(), ckpt_path))
            print(f'[{run_name}] new best val_mIoU={best_miou:.4f}, checkpoint saved.')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'[{run_name}] no improvement for {patience} epochs, stopping early at epoch {epoch+1}.')
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model, history, best_miou


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def release_model(model):
    """Explicitly frees a finished run's model before building the next one.
    With 12 training runs happening back-to-back in one notebook kernel, we
    tried just letting Python's garbage collector catch each old model on its
    own and ran into memory creeping up over many runs, so we free it right
    away instead of waiting."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print('Training loop ready. Device:', device)


Training loop ready. Device: cuda


**Step A:** Pilot test, 5 encoder options on U-Net only. Best one carries forward to Step B.

In [16]:
train_ds = PatchDataset(train_df, PATCH_FOLDER, augment=True, oversample_rare=True)
val_ds = PatchDataset(val_df, PATCH_FOLDER, augment=False)
train_ds_3ch = PatchDataset(train_df, PATCH_FOLDER, augment=True, oversample_rare=True, add_vv_vh_diff=True)
val_ds_3ch = PatchDataset(val_df, PATCH_FOLDER, augment=False, add_vv_vh_diff=True)

loss_fn = CombinedLoss(class_weights=class_weights)

PILOT_CONFIGS = [
    {'name': 'unet_ssl4eo_moco_finetune', 'encoder_source': 'ssl4eo_moco', 'freeze_encoder': False},
    {'name': 'unet_ssl4eo_moco_frozen', 'encoder_source': 'ssl4eo_moco', 'freeze_encoder': True},
    {'name': 'unet_softcon_finetune', 'encoder_source': 'softcon', 'freeze_encoder': False},
    {'name': 'unet_imagenet_finetune', 'encoder_source': 'imagenet', 'freeze_encoder': False},
    {'name': 'unet_random_init', 'encoder_source': 'random', 'freeze_encoder': False},
]

pilot_results = {}
for cfg in PILOT_CONFIGS:
    print(f"\n=== Pilot run: {cfg['name']} ===")
    model = build_model('Unet', cfg['encoder_source'], num_classes=NUM_CLASSES,
                         in_channels=2, freeze_encoder=cfg['freeze_encoder'])
    trained_model, history, best_miou = train_one_run(
        cfg['name'], model, train_ds, val_ds, loss_fn, device,
        batch_size=24, max_epochs=30, patience=6, checkpoint_dir=CKPT_DIR,
    )
    pilot_results[cfg['name']] = {'best_val_mIoU': best_miou, 'history': history}
    release_model(trained_model)

print('\nPilot results (best validation mIoU):')
for name, res in sorted(pilot_results.items(), key=lambda kv: -kv[1]['best_val_mIoU']):
    print(f"  {name}: {res['best_val_mIoU']:.4f}")


=== Pilot run: unet_ssl4eo_moco_finetune ===
Downloading: "https://hf.co/torchgeo/resnet50_sentinel1_all_moco/resolve/e79862c667853c10a709bdd77ea8ffbad0e0f1cf/resnet50_sentinel1_all_moco-906e4356.pth" to /root/.cache/torch/hub/checkpoints/resnet50_sentinel1_all_moco-906e4356.pth


100%|██████████| 90.0M/90.0M [00:02<00:00, 41.8MB/s]


[unet_ssl4eo_moco_finetune] Resuming: 13 epoch(s) already done.
[unet_ssl4eo_moco_finetune] epoch 14/30 step 50/887 loss=0.7234
[unet_ssl4eo_moco_finetune] epoch 14/30 step 100/887 loss=0.7259
[unet_ssl4eo_moco_finetune] epoch 14/30 step 150/887 loss=0.7260
[unet_ssl4eo_moco_finetune] epoch 14/30 step 200/887 loss=0.7251
[unet_ssl4eo_moco_finetune] epoch 14/30 step 250/887 loss=0.7238
[unet_ssl4eo_moco_finetune] epoch 14/30 step 300/887 loss=0.7239
[unet_ssl4eo_moco_finetune] epoch 14/30 step 350/887 loss=0.7227
[unet_ssl4eo_moco_finetune] epoch 14/30 step 400/887 loss=0.7202
[unet_ssl4eo_moco_finetune] epoch 14/30 step 450/887 loss=0.7183
[unet_ssl4eo_moco_finetune] epoch 14/30 step 500/887 loss=0.7164
[unet_ssl4eo_moco_finetune] epoch 14/30 step 550/887 loss=0.7147
[unet_ssl4eo_moco_finetune] epoch 14/30 step 600/887 loss=0.7125
[unet_ssl4eo_moco_finetune] epoch 14/30 step 650/887 loss=0.7101
[unet_ssl4eo_moco_finetune] epoch 14/30 step 700/887 loss=0.7085
[unet_ssl4eo_moco_finetune]

100%|██████████| 90.0M/90.0M [00:04<00:00, 20.5MB/s]


[unet_softcon_finetune] Resuming: 12 epoch(s) already done.
[unet_softcon_finetune] epoch 13/30 step 50/887 loss=0.5095
[unet_softcon_finetune] epoch 13/30 step 100/887 loss=0.5111
[unet_softcon_finetune] epoch 13/30 step 150/887 loss=0.5092
[unet_softcon_finetune] epoch 13/30 step 200/887 loss=0.5076
[unet_softcon_finetune] epoch 13/30 step 250/887 loss=0.5085
[unet_softcon_finetune] epoch 13/30 step 300/887 loss=0.5077
[unet_softcon_finetune] epoch 13/30 step 350/887 loss=0.5071
[unet_softcon_finetune] epoch 13/30 step 400/887 loss=0.5068
[unet_softcon_finetune] epoch 13/30 step 450/887 loss=0.5057
[unet_softcon_finetune] epoch 13/30 step 500/887 loss=0.5055
[unet_softcon_finetune] epoch 13/30 step 550/887 loss=0.5047
[unet_softcon_finetune] epoch 13/30 step 600/887 loss=0.5043
[unet_softcon_finetune] epoch 13/30 step 650/887 loss=0.5037
[unet_softcon_finetune] epoch 13/30 step 700/887 loss=0.5036
[unet_softcon_finetune] epoch 13/30 step 750/887 loss=0.5035
[unet_softcon_finetune] ep

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[unet_imagenet_finetune] Resuming: 9 epoch(s) already done.
[unet_imagenet_finetune] epoch 10/30 step 50/887 loss=0.6448
[unet_imagenet_finetune] epoch 10/30 step 100/887 loss=0.6428
[unet_imagenet_finetune] epoch 10/30 step 150/887 loss=0.6410
[unet_imagenet_finetune] epoch 10/30 step 200/887 loss=0.6405
[unet_imagenet_finetune] epoch 10/30 step 250/887 loss=0.6392
[unet_imagenet_finetune] epoch 10/30 step 300/887 loss=0.6389
[unet_imagenet_finetune] epoch 10/30 step 350/887 loss=0.6382
[unet_imagenet_finetune] epoch 10/30 step 400/887 loss=0.6372
[unet_imagenet_finetune] epoch 10/30 step 450/887 loss=0.6369
[unet_imagenet_finetune] epoch 10/30 step 500/887 loss=0.6367
[unet_imagenet_finetune] epoch 10/30 step 550/887 loss=0.6359
[unet_imagenet_finetune] epoch 10/30 step 600/887 loss=0.6355
[unet_imagenet_finetune] epoch 10/30 step 650/887 loss=0.6352
[unet_imagenet_finetune] epoch 10/30 step 700/887 loss=0.6344
[unet_imagenet_finetune] epoch 10/30 step 750/887 loss=0.6335
[unet_image

**Winner selection:** pick the pilot config with the best validation mIoU.

In [17]:
winner_name = max(pilot_results, key=lambda k: pilot_results[k]['best_val_mIoU'])
winner_cfg = next(c for c in PILOT_CONFIGS if c['name'] == winner_name)
WINNER_ENCODER_SOURCE = winner_cfg['encoder_source']
WINNER_FREEZE_ENCODER = winner_cfg['freeze_encoder']

print(f'Winning pilot config: {winner_name}')
print(f'  encoder_source = {WINNER_ENCODER_SOURCE}')
print(f'  freeze_encoder = {WINNER_FREEZE_ENCODER}')
print(f"  best val_mIoU  = {pilot_results[winner_name]['best_val_mIoU']:.4f}")


Winning pilot config: unet_ssl4eo_moco_finetune
  encoder_source = ssl4eo_moco
  freeze_encoder = False
  best val_mIoU  = 0.5081
